# Exercise — Configure Governed Access in Lake Formation

**Trailhead Provisions** needs four personas to have exactly the access below on the `customer`
table. Configure the Lake Formation grants with the provided helpers, then run the verification
cell to prove enforcement by querying as each persona in Athena. See `INSTRUCTIONS.md`.

| Persona | Intended access | LF mode |
|---|---|---|
| `co_marketing_analyst` | Customer **minus** `email`, `phone` | column-level |
| `co_cs_rep` | All columns, **rows where `region='EU'`** only | row-level |
| `co_data_steward` | Everything tagged `governed=true` | tag-based (LF-TBAC) |
| `co_auditor` | Schema + tags only; **no row data** | metadata-only |

> Runs against live AWS Lake Formation + Athena when provisioned; local fallback offline.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, verify_step7, PERSONAS

gc = make_catalog("trailhead.db")
print("Backend:", lf_backend(), "->", type(gc).__name__)

## 1. Confirm the starting state (provided)
No grants yet — every persona has no access.

In [ ]:
gc.grants_summary()

## 2. Apply the four Lake Formation grants
Use the provided helpers: `grant_columns(role, table, exclude=[...])`, `create_row_filter(name, table, expr)` + `grant_rows(role, table, name)`, `assign_tag(table, k, v)` + `grant_by_tag(role, k, v)`, and `grant_describe_only(role)`.

In [ ]:
# 1) column-level: analyst sees customer minus PII
gc.grant_columns("co_marketing_analyst", "customer", exclude=["email", "phone"])

# 2) row-level: cs_rep sees only EU rows
gc.create_row_filter("eu_only", "customer", "region = 'EU'")
gc.grant_rows("co_cs_rep", "customer", "eu_only")

# 3) tag-based: tag the customer table governed=true, grant the steward by tag
gc.assign_tag("customer", "governed", "true")
gc.grant_by_tag("co_data_steward", "governed", "true")

# 4) metadata-only: auditor gets DESCRIBE, no SELECT
gc.grant_describe_only("co_auditor")

gc.grants_summary()

## 3. Verify enforcement (provided)
Queries the governed table as each persona and asserts the intended outcome. Every check must read **PASS**.

In [ ]:
results = verify_step7(gc)
print("All checks pass?", (results["result"] == "PASS").all())
results

## 4. Policy spec
Replace the cell below with your short policy spec. Address every requirement in `INSTRUCTIONS.md`.

### Policy spec — Trailhead customer access

- **`co_marketing_analyst`** targets campaigns and must never see raw contact PII. **Column-level**
  grant on `customer` excluding `email`, `phone`; all non-PII columns remain available.
- **`co_cs_rep`** handles EU tickets only: full customer detail but **only `region='EU'` rows**,
  via a **row-level** data-cell filter. No access to other regions' customers.
- **`co_data_steward`** owns customer-domain quality: full access via an **LF-Tag** grant on
  `governed=true`, so any future table tagged governed is covered automatically — not per-table.
- **`co_auditor`** verifies posture and must **not** read customer data: **metadata-only**
  (DESCRIBE / schema + tags), SELECT denied. Least privilege — proves governance exists without
  ever touching a personal record.

Each grant is the smallest privilege that satisfies the role's job; the auditor's metadata-only
access is the clearest example — visibility into structure with zero exposure of personal data.